# Utilization telemetry — NVML

In [2]:
import pynvml
import time
import pandas as pd

pynvml.nvmlInit()
N = pynvml.nvmlDeviceGetCount()
handles = [pynvml.nvmlDeviceGetHandleByIndex(i) for i in range(N)]
print(f"{N} GPUs | driver {pynvml.nvmlSystemGetDriverVersion()}")

8 GPUs | driver 580.126.16


## SM + memory controller utilization
`nvmlDeviceGetUtilizationRates` returns two numbers:
- `gpu` — % of time *any* SM warp was active over the last ~1/6s sample window. This is the headline compute signal.
- `memory` — % of time the memory controller had an outstanding transaction. Tells you memory bus pressure, not how much memory is *used* (that's separate).

Both are coarse time-averaged percentages, not instantaneous. On an idle GPU expect ~0% for both.

In [3]:
def get_utilization(handles):
    rows = []
    for i, h in enumerate(handles):
        u = pynvml.nvmlDeviceGetUtilizationRates(h)
        rows.append({"gpu": i, "sm_pct": u.gpu, "mem_ctrl_pct": u.memory})
    return pd.DataFrame(rows).set_index("gpu")

get_utilization(handles)

,sm_pct,mem_ctrl_pct
gpu,,
0,0,0
1,0,0
2,0,0
3,0,0
4,0,0
5,0,0
6,0,0
7,0,0


## Encoder / decoder / specialized engines
A100 has dedicated fixed-function hardware for video encode (NVENC), video decode (NVDEC), and JPEG (NVJPEG). These run independently of the SMs.

- Encoder/decoder util returns `(utilization_pct, sampling_period_us)` — the sampling period varies.
- These will be 0 on a training/inference workload — they only light up for video transcoding pipelines.
- Worth collecting anyway: non-zero encoder util during a supposed training run is anomalous and worth flagging.

In [4]:
def get_engine_utilization(handles):
    rows = []
    for i, h in enumerate(handles):
        enc_pct, enc_period = pynvml.nvmlDeviceGetEncoderUtilization(h)
        dec_pct, dec_period = pynvml.nvmlDeviceGetDecoderUtilization(h)
        rows.append({
            "gpu": i,
            "encoder_pct": enc_pct, "enc_sample_us": enc_period,
            "decoder_pct": dec_pct, "dec_sample_us": dec_period,
        })
    return pd.DataFrame(rows).set_index("gpu")

get_engine_utilization(handles)

,encoder_pct,enc_sample_us,decoder_pct,dec_sample_us
gpu,,,,
0,0,200000,0,200000
1,0,200000,0,200000
2,0,200000,0,200000
3,0,200000,0,200000
4,0,200000,0,200000
5,0,200000,0,200000
6,0,200000,0,200000
7,0,200000,0,200000


## Clock speeds — current and max
Four clock domains queryable: SM (compute), memory, graphics (same as SM on A100), video.

Current clocks drop when throttled. Comparing current vs. max tells you if/how much the GPU is being held back.
`nvmlDeviceGetApplicationsClock` returns the clock that was set via app-clock locking (if any).

In [5]:
CLOCK_TYPES = {
    "sm":       pynvml.NVML_CLOCK_SM,
    "mem":      pynvml.NVML_CLOCK_MEM,
    "graphics": pynvml.NVML_CLOCK_GRAPHICS,
    "video":    pynvml.NVML_CLOCK_VIDEO,
}

def get_clocks(handles):
    rows = []
    for i, h in enumerate(handles):
        row = {"gpu": i}
        for name, ctype in CLOCK_TYPES.items():
            row[f"{name}_cur_MHz"] = pynvml.nvmlDeviceGetClockInfo(h, ctype)
            row[f"{name}_max_MHz"] = pynvml.nvmlDeviceGetMaxClockInfo(h, ctype)
        rows.append(row)
    return pd.DataFrame(rows).set_index("gpu")

get_clocks(handles)

,sm_cur_MHz,sm_max_MHz,mem_cur_MHz,mem_max_MHz,graphics_cur_MHz,graphics_max_MHz,video_cur_MHz,video_max_MHz
gpu,,,,,,,,
0,210,1410,1593,1593,210,1410,585,1290
1,210,1410,1593,1593,210,1410,585,1290
2,210,1410,1593,1593,210,1410,585,1290
3,210,1410,1593,1593,210,1410,585,1290
4,210,1410,1593,1593,210,1410,585,1290
5,210,1410,1593,1593,210,1410,585,1290
6,210,1410,1593,1593,210,1410,585,1290
7,210,1410,1593,1593,210,1410,585,1290


## Running processes
Lists every process currently using the GPU, with PID and how much framebuffer memory it's holding.

Three separate queries:
- **compute** — CUDA processes (training, inference, anything using the CUDA runtime)
- **graphics** — OpenGL/Vulkan processes (rare on A100, but check)
- **MPS** — processes running under NVIDIA Multi-Process Service (shared CUDA context)

Memory used per process is a strong signal: a training job holds weights + gradients + optimizer states (Adam ≈ 8× params in fp32). An inference job holds weights + KV cache only.

In [ ]:
import os

def _proc_name(pid):
    try:
        with open(f"/proc/{pid}/cmdline", "rb") as f:
            return f.read().replace(b"\x00", b" ").decode(errors="replace").strip()[:80]
    except Exception:
        return "?"

def get_processes(handles):
    rows = []
    for i, h in enumerate(handles):
        for kind, fn in [("compute", pynvml.nvmlDeviceGetComputeRunningProcesses),
                         ("graphics", pynvml.nvmlDeviceGetGraphicsRunningProcesses),
                         ("mps", pynvml.nvmlDeviceGetMPSComputeRunningProcesses)]:
            try:
                procs = fn(h)
            except pynvml.NVMLError:
                continue
            for p in procs:
                rows.append({
                    "gpu": i, "kind": kind, "pid": p.pid,
                    "mem_MiB": p.usedGpuMemory // 1024**2 if p.usedGpuMemory else 0,
                    "cmdline": _proc_name(p.pid),
                })
    if not rows:
        print("No processes using GPUs")
        return pd.DataFrame()
    return pd.DataFrame(rows)

get_processes(handles)

## Historical samples buffer
NVML maintains an internal ring buffer of recent samples for GPU util, memory util, encoder util, decoder util, and power. You can pull the whole buffer in one call — useful for getting a burst of history without having polled yourself.

`nvmlDeviceGetSamples(handle, sample_type, last_seen_timestamp)` returns `(value_type, [samples])`. Pass `lastSeenTimeStamp=0` to get everything in the buffer (typically ~100 samples, ~10s of history at 1/10s resolution).

In [ ]:
SAMPLE_TYPES = {
    "gpu_util":   pynvml.NVML_GPU_UTILIZATION_SAMPLES,
    "mem_util":   pynvml.NVML_MEMORY_UTILIZATION_SAMPLES,
    "enc_util":   pynvml.NVML_ENC_UTILIZATION_SAMPLES,
    "dec_util":   pynvml.NVML_DEC_UTILIZATION_SAMPLES,
    "power_mw":   pynvml.NVML_POWER_SAMPLES,
}

def get_sample_buffer(handle, gpu_idx=0):
    """Pull all buffered historical samples for one GPU. Returns a DataFrame indexed by timestamp (us)."""
    dfs = []
    for name, stype in SAMPLE_TYPES.items():
        try:
            _, samples = pynvml.nvmlDeviceGetSamples(handle, stype, 0)
            df = pd.DataFrame([{"ts_us": s.timeStamp, name: s.sampleValue.uiVal} for s in samples])
            df = df.set_index("ts_us")
            dfs.append(df)
        except pynvml.NVMLError as e:
            print(f"  {name}: unavailable ({e})")
    if not dfs:
        return pd.DataFrame()
    result = dfs[0].join(dfs[1:], how="outer").sort_index()
    if "power_mw" in result.columns:
        result["power_W"] = result["power_mw"] / 1000
    return result

# Pull for GPU 0 — change index to inspect others
buf = get_sample_buffer(handles[0], gpu_idx=0)
print(f"{len(buf)} samples, spanning {(buf.index[-1] - buf.index[0]) / 1e6:.1f}s")
buf.tail(10)

## Violation status — time spent throttled
`nvmlDeviceGetViolationStatus` returns cumulative counters (in microseconds) of how long the GPU has spent being held back by each policy since last driver load. Diff two readings to get a rate.

Policies:
- `POWER` — time throttled due to power cap
- `THERMAL` — time throttled due to temperature
- `SYNC_BOOST` — time held back for sync boost alignment
- `BOARD_LIMIT` — board-level power limit
- `LOW_UTILIZATION` — voluntarily underclocked due to low load
- `RELIABILITY` — reliability voltage limit

In [ ]:
PERF_POLICIES = {
    "power":           pynvml.NVML_PERF_POLICY_POWER,
    "thermal":         pynvml.NVML_PERF_POLICY_THERMAL,
    "sync_boost":      pynvml.NVML_PERF_POLICY_SYNC_BOOST,
    "board_limit":     pynvml.NVML_PERF_POLICY_BOARD_LIMIT,
    "low_utilization": pynvml.NVML_PERF_POLICY_LOW_UTILIZATION,
    "reliability":     pynvml.NVML_PERF_POLICY_RELIABILITY,
}

def get_violation_status(handles):
    rows = []
    for i, h in enumerate(handles):
        row = {"gpu": i}
        for name, policy in PERF_POLICIES.items():
            try:
                v = pynvml.nvmlDeviceGetViolationStatus(h, policy)
                # referenceTime = total elapsed us, violationTime = us spent throttled
                row[f"{name}_throttled_us"] = v.violationTime
                row[f"{name}_pct"] = round(100 * v.violationTime / v.referenceTime, 2) if v.referenceTime else 0
            except pynvml.NVMLError:
                row[f"{name}_throttled_us"] = None
                row[f"{name}_pct"] = None
        rows.append(row)
    return pd.DataFrame(rows).set_index("gpu")

get_violation_status(handles)

## Live utilization monitor
Polls SM util, memory controller util, and current SM clock across all GPUs. Returns a DataFrame for further analysis.

In [ ]:
def live_utilization(handles, duration=10, interval=1):
    header = f"{'t':>2} | " + "  ".join(f"G{i}sm% G{i}mem%" for i in range(len(handles)))
    records = []
    for t in range(duration):
        row = {"t": t}
        parts = []
        for i, h in enumerate(handles):
            u = pynvml.nvmlDeviceGetUtilizationRates(h)
            clk = pynvml.nvmlDeviceGetClockInfo(h, pynvml.NVML_CLOCK_SM)
            row[f"gpu{i}_sm_pct"] = u.gpu
            row[f"gpu{i}_mem_pct"] = u.memory
            row[f"gpu{i}_sm_MHz"] = clk
            parts.append(f"{u.gpu:>4}% {u.memory:>4}%")
        records.append(row)
        if t == 0:
            print(header)
            print("-" * len(header))
        print(f"{t:>2} | " + "  ".join(parts))
        if t < duration - 1:
            time.sleep(interval)
    return pd.DataFrame(records).set_index("t")

df_util = live_utilization(handles, duration=10)

---
## DCGM — setup
pydcgm lives at `/usr/local/dcgm/bindings/python3/` (not on PyPI). We add it to the path and connect to the already-running `nv-hostengine` daemon.

Key objects:
- `DcgmHandle` — connection to the daemon
- `DcgmGroup` — a set of GPUs to query together (default = all 8)
- `DcgmFieldGroup` — a named set of field IDs to watch
- `WatchFields` — tells the daemon to start sampling at a given interval
- `GetLatestValues` — pull the most recent sample for each field × GPU

In [ ]:
import sys
sys.path.insert(0, "/usr/local/dcgm/bindings/python3")

import pydcgm
import dcgm_fields
import dcgm_structs

dcgm_handle = pydcgm.DcgmHandle(ipAddress="127.0.0.1")
dcgm_system = pydcgm.DcgmSystem(dcgm_handle)
dcgm_group  = pydcgm.DcgmGroup(dcgm_handle, groupName="all_gpus",
                                groupType=dcgm_structs.DCGM_GROUP_DEFAULT)
print("Connected. GPUs:", dcgm_group.GetGpuIds())

## DCGM standard utilization fields
These are always available — no PMU exclusivity required.

| Field | What it measures |
|---|---|
| `GR_ENGINE_ACTIVE` | Fraction of time the graphics/compute engine had work queued. Coarser than SM_ACTIVE. |
| `SM_ACTIVE` | Fraction of SMs with ≥1 active warp. Better than NVML's utilization rate. |
| `SM_OCCUPANCY` | Active warps / max possible warps, averaged across SMs. Tells you how well-packed the SMs are. |
| `DRAM_ACTIVE` | Fraction of cycles HBM had an active read or write transaction. Memory bandwidth pressure. |

Values are 0.0–1.0 (not percent). Multiply by 100 if you want %.

In [ ]:
STANDARD_FIELDS = {
    "gr_engine_active": dcgm_fields.DCGM_FI_PROF_GR_ENGINE_ACTIVE,
    "sm_active":        dcgm_fields.DCGM_FI_PROF_SM_ACTIVE,
    "sm_occupancy":     dcgm_fields.DCGM_FI_PROF_SM_OCCUPANCY,
    "dram_active":      dcgm_fields.DCGM_FI_PROF_DRAM_ACTIVE,
}

def dcgm_snapshot(handle, group, fields_dict, watch_interval_us=100_000, label="fields"):
    """Watch a set of fields, wait one interval, pull latest values. Returns DataFrame."""
    field_ids   = list(fields_dict.values())
    field_names = list(fields_dict.keys())
    gpu_ids     = group.GetGpuIds()

    fg = pydcgm.DcgmFieldGroup(handle, label, field_ids)
    group.samples.WatchFields(fg, watch_interval_us, maxKeepAge=60.0, maxKeepSamples=10)
    time.sleep(watch_interval_us / 1e6 * 2)  # wait for at least one sample

    latest = group.samples.GetLatestValues(fg)
    rows = []
    for gpu_id in gpu_ids:
        row = {"gpu": gpu_id}
        for name, fid in fields_dict.items():
            val = latest[fid][gpu_id].value
            # DCGM uses sentinel values for "not available"
            row[name] = round(val, 4) if val < 1e30 else None
        rows.append(row)

    group.samples.UnwatchFields(fg)
    fg.Delete()
    return pd.DataFrame(rows).set_index("gpu")

dcgm_snapshot(dcgm_handle, dcgm_group, STANDARD_FIELDS, label="standard")

## DCGM profiling fields — compute pipeline
These tap the hardware PMU. **Only one consumer at a time** — will fail/return garbage if Nsight Compute is running simultaneously.

| Field | What it measures | Why it matters |
|---|---|---|
| `PIPE_TENSOR_ACTIVE` | Fraction of cycles any Tensor Core op was executing | The single strongest training signal |
| `PIPE_TENSOR_HMMA_ACTIVE` | Tensor Core cycles doing half-precision matmul (fp16/bf16) | The specific op training uses |
| `PIPE_TENSOR_IMMA_ACTIVE` | Tensor Core cycles doing integer matmul | Quantized inference |
| `PIPE_TENSOR_DFMA_ACTIVE` | Tensor Core cycles doing FP64 matmul | HPC / scientific, not training |
| `PIPE_FP32_ACTIVE` | CUDA core FP32 pipe | Non-tensor compute |
| `PIPE_FP64_ACTIVE` | CUDA core FP64 pipe | HPC |
| `PIPE_FP16_ACTIVE` | CUDA core FP16 pipe (non-Tensor) | Rare |
| `PIPE_INT_ACTIVE` | Integer pipe | Address calc, control flow |

Training workloads: `HMMA_ACTIVE` very high, `IMMA` near zero, `FP64` near zero.

In [ ]:
PROFILING_FIELDS = {
    "tensor_active":      dcgm_fields.DCGM_FI_PROF_PIPE_TENSOR_ACTIVE,
    "tensor_hmma":        dcgm_fields.DCGM_FI_PROF_PIPE_TENSOR_HMMA_ACTIVE,
    "tensor_imma":        dcgm_fields.DCGM_FI_PROF_PIPE_TENSOR_IMMA_ACTIVE,
    "tensor_dfma":        dcgm_fields.DCGM_FI_PROF_PIPE_TENSOR_DFMA_ACTIVE,
    "fp32_active":        dcgm_fields.DCGM_FI_PROF_PIPE_FP32_ACTIVE,
    "fp64_active":        dcgm_fields.DCGM_FI_PROF_PIPE_FP64_ACTIVE,
    "fp16_active":        dcgm_fields.DCGM_FI_PROF_PIPE_FP16_ACTIVE,
    "int_active":         dcgm_fields.DCGM_FI_PROF_PIPE_INT_ACTIVE,
}

# Note: if this errors with "not supported" the PMU is held by another process
dcgm_snapshot(dcgm_handle, dcgm_group, PROFILING_FIELDS, label="profiling")

## DCGM live profiling monitor
Polls both standard + profiling fields together at a set interval. This is what you'd run during a stress test to see the pipeline light up.

In [ ]:
ALL_PROF_FIELDS = {**STANDARD_FIELDS, **PROFILING_FIELDS}

def live_dcgm(handle, group, fields_dict, duration=10, interval_s=1):
    """Poll DCGM fields every interval_s for duration seconds. Returns DataFrame."""
    field_ids = list(fields_dict.values())
    gpu_ids   = group.GetGpuIds()

    fg = pydcgm.DcgmFieldGroup(handle, "live_monitor", field_ids)
    group.samples.WatchFields(fg, int(interval_s * 1e6), maxKeepAge=120.0, maxKeepSamples=200)
    time.sleep(interval_s)  # first sample warmup

    records = []
    for t in range(duration):
        latest = group.samples.GetLatestValues(fg)
        for gpu_id in gpu_ids:
            row = {"t": t, "gpu": gpu_id}
            for name, fid in fields_dict.items():
                val = latest[fid][gpu_id].value
                row[name] = round(val, 4) if val < 1e30 else None
            records.append(row)

        # compact print: one line per tick showing GPU0 as representative
        g0 = {name: latest[fid][0].value for name, fid in fields_dict.items()}
        sm  = g0.get("sm_active", 0) or 0
        ten = g0.get("tensor_active", 0) or 0
        hmm = g0.get("tensor_hmma", 0) or 0
        drm = g0.get("dram_active", 0) or 0
        print(f"t={t:>2} GPU0 | sm_active={sm:.3f}  tensor={ten:.3f}  hmma={hmm:.3f}  dram={drm:.3f}")

        if t < duration - 1:
            time.sleep(interval_s)

    group.samples.UnwatchFields(fg)
    fg.Delete()
    return pd.DataFrame(records)

df_dcgm = live_dcgm(dcgm_handle, dcgm_group, ALL_PROF_FIELDS, duration=10)